# Common setup


Outputs in data/processed:
- features.parquet: row_id, NPI, peer keys, 4 features
- peers_30.parquet, peers_50.parquet, peers_100.parquet: row_id, peer_level, peer_id, peer_n, robust z

In [ ]:
import numpy as np
import pandas as pd
import os

cols = ["Rndrng_NPI", "Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc", "Rndrng_Prvdr_Cntry",
        "Tot_Benes", "Tot_Srvcs", "Tot_Bene_Day_Srvcs",
        "Avg_Sbmtd_Chrg", "Avg_Mdcr_Alowd_Amt", "Avg_Mdcr_Stdzd_Amt"]
df = pd.read_parquet("medicare_2024.parquet", columns=cols)

# same cleaning as data_preprocessing.ipynb
keep = ((df["Rndrng_Prvdr_Cntry"] == "US")
        & (df["Tot_Srvcs"] >= df["Tot_Benes"])
        & (df["Tot_Bene_Day_Srvcs"] <= df["Tot_Srvcs"])
        & (df["Avg_Mdcr_Alowd_Amt"] > 0))
df = df[keep].reset_index(drop=True)
df["row_id"] = np.arange(len(df))
len(df)

9781161

In [ ]:
df["log_pay"] = np.log(df["Avg_Mdcr_Stdzd_Amt"].clip(lower=1e-6))  # payment level (avoid log(0))
df["log_benes"] = np.log(df["Tot_Benes"])  # volume
df["srvcs_per_bene"] = np.log(df["Tot_Srvcs"] / df["Tot_Benes"])  # intensity
df["markup"] = np.log(df["Avg_Sbmtd_Chrg"].clip(lower=1e-6) / df["Avg_Mdcr_Alowd_Amt"])  # charge vs allowed

feats = ["log_pay", "log_benes", "srvcs_per_bene", "markup"]
rz_cols = [f + "_rz" for f in feats]

## Peer groups: id and size on every level

In [ ]:
levels = {1: ["Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc"], #Level 1
          2: ["HCPCS_Cd", "Place_Of_Srvc"], #Level 2
          3: ["HCPCS_Cd"]} #Level 3

for lvl, keys in levels.items():
    df[f"gid{lvl}"] = df.groupby(keys, observed=True).ngroup() + lvl * 10_000_000  # group id, 10M apart so levels don't overlap
    df[f"size{lvl}"] = df.groupby(keys, observed=True)["row_id"].transform("size")  # number of rows in the group

## Robust z on every level

In [ ]:
def robust_z(df, keys):
    med = df.groupby(keys, observed=True)[feats].transform("median")  # group median
    dev = (df[feats] - med).abs()  # distance from the median
    dev[keys] = df[keys]  # add group columns so we can group the deviations
    mad = dev.groupby(keys, observed=True)[feats].transform("median")
    mean_dev = dev.groupby(keys, observed=True)[feats].transform("mean")
    scale = (1.4826 * mad).where(mad > 0, 1.2533 * mean_dev)  # if MAD = 0, use mean abs deviation
    return ((df[feats] - med) / scale.where(scale > 0)).astype("float32")

rz_levels = {}
for lvl, keys in levels.items():
    rz_levels[lvl] = robust_z(df, keys)  # z-scores if every row used this level

## Pick the level for each row and save one file per min size

In [ ]:
def make_peers(df, min_size):
    p = df[["row_id"]].copy()
    p["peer_level"] = 0  # 0 = unscored
    p["peer_id"] = -1
    p["peer_n"] = 0
    for lvl in [1, 2, 3]:
        take = (p["peer_level"] == 0) & (df[f"size{lvl}"] >= min_size)  # not assigned yet + group is big enough
        p.loc[take, "peer_level"] = lvl
        p.loc[take, "peer_id"] = df.loc[take, f"gid{lvl}"]
        p.loc[take, "peer_n"] = df.loc[take, f"size{lvl}"]

    p[rz_cols] = np.nan  # unscored rows stay NaN
    for lvl in [1, 2, 3]:
        rows = p["peer_level"] == lvl
        p.loc[rows, rz_cols] = rz_levels[lvl].loc[rows].to_numpy()  # take z-scores from the chosen level
    p[rz_cols] = p[rz_cols].astype("float32")
    return p

In [ ]:
os.makedirs("data/processed", exist_ok=True)
coverage = []

for m in [30, 50, 100]:
    p = make_peers(df, m)
    p.to_parquet(f"data/processed/peers_{m}.parquet", index=False)

    lv = p["peer_level"].value_counts(normalize=True) * 100  # % of rows on each level
    coverage.append({"min_size": m,
                     "level 1 %": lv.get(1, 0),
                     "level 2 %": lv.get(2, 0),
                     "level 3 %": lv.get(3, 0),
                     "unscored %": lv.get(0, 0)})

coverage = pd.DataFrame(coverage).set_index("min_size").round(2)
coverage

,level 1 %,level 2 %,level 3 %,unscored %
min_size,,,,
30,97.63,2.03,0.10,0.24
50,96.63,2.82,0.16,0.39
100,94.56,4.36,0.29,0.78


In [ ]:
# shared features file, same for every min size
keep_cols = ["row_id", "Rndrng_NPI", "Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc"] + feats
df[keep_cols].to_parquet("data/processed/features.parquet", index=False)